# Waste Classification and Recycling Optimization Using Agentic AI and Computer Vision
## Part 1 - Training the vision model (ResNet50)

This notebook trains the computer-vision model that the agent uses as a tool. Run all cells top to bottom (GPU: Runtime > Change runtime type > T4).

**What is different from a basic notebook:** a stratified, seeded 70/15/15 train/val/test split; validation and test images are never augmented; the best checkpoint is chosen on validation only; the test set is scored exactly once at the end. Classes: cardboard, glass, metal, paper, plastic, trash.

## 1. Setup and data

In [ ]:
import os, json, random, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

SEED, EPOCHS, BATCH, LR = 42, int(os.getenv("EPOCHS", 15)), 32, 1e-4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

try:  # Google Colab: dataset lives in Drive under 'new datasets ' (trailing space), one folder per class
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = "/content/drive/MyDrive/new datasets "
    DATA_DIR = Path("/content/dataset")
    if not DATA_DIR.exists():
        shutil.copytree(DRIVE_DATA, DATA_DIR)
    OUT_DIR = Path("/content/weights")
except ImportError:  # running locally
    DATA_DIR, OUT_DIR = Path("data/raw"), Path("weights_nb")
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
EXTS = {".jpg", ".jpeg", ".png"}
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
items = [(p, classes.index(d.name)) for d in DATA_DIR.iterdir() if d.is_dir()
         for p in sorted(d.iterdir()) if p.suffix.lower() in EXTS]
labels = [l for _, l in items]
counts = np.bincount(labels, minlength=len(classes))
print(dict(zip(classes, counts.tolist())), "total:", len(items))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(classes, counts, color="#2f7d46"); ax[0].set_title("Images per class"); ax[0].tick_params(axis="x", rotation=30)
ax[1].pie(counts, labels=classes, autopct="%1.1f%%"); ax[1].set_title("Dataset distribution")
plt.tight_layout(); plt.show()

## 2. Split (70/15/15, stratified, seeded) and data loaders

In [ ]:
train_i, rest_i = train_test_split(range(len(items)), test_size=0.30, stratify=labels, random_state=SEED)
val_i, test_i = train_test_split(rest_i, test_size=0.50, stratify=[labels[i] for i in rest_i], random_state=SEED)
print(f"train={len(train_i)} val={len(val_i)} test={len(test_i)}")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.3, 0.3, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImageList(Dataset):
    def __init__(self, idx, tf): self.items, self.tf = [items[i] for i in idx], tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]; return self.tf(Image.open(p).convert("RGB")), y

mk = lambda idx, tf, shuffle: DataLoader(ImageList(idx, tf), batch_size=BATCH, shuffle=shuffle, num_workers=0 if os.name == "nt" else 2)
train_dl, val_dl, test_dl = mk(train_i, train_tf, True), mk(val_i, eval_tf, False), mk(test_i, eval_tf, False)

## 3. Model: ResNet50 pretrained on ImageNet, new 6-way head

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model.to(device)

tr_counts = np.bincount([labels[i] for i in train_i], minlength=len(classes))
class_w = torch.tensor(tr_counts.sum() / (len(classes) * tr_counts), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)   # compensates for class imbalance
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 4. Train (best checkpoint chosen on validation only, early stopping)

In [ ]:
def evaluate(loader):
    model.eval(); preds, ys, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device); out = model(x)
            loss_sum += criterion(out, y).item() * len(y)
            preds += out.argmax(1).cpu().tolist(); ys += y.cpu().tolist()
    return loss_sum / len(ys), float(np.mean(np.array(preds) == np.array(ys))), ys, preds

history, best_acc, best_epoch, patience = [], 0.0, 0, 0
for epoch in range(1, EPOCHS + 1):
    model.train(); run = 0.0; correct = 0
    for x, y in tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(device), y.to(device); optimizer.zero_grad()
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            out = model(x); loss = criterion(out, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        run += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item()
    scheduler.step()
    vl, va, _, _ = evaluate(val_dl)
    history.append(dict(epoch=epoch, train_loss=run / len(train_i), train_acc=correct / len(train_i), val_loss=vl, val_acc=va))
    print(f"epoch {epoch:2d} | train_loss {history[-1]['train_loss']:.3f} acc {history[-1]['train_acc']:.3f} | val_loss {vl:.3f} acc {va:.3f}")
    if va > best_acc:
        best_acc, best_epoch, patience = va, epoch, 0
        torch.save(model.state_dict(), OUT_DIR / "best_model.pth")
    else:
        patience += 1
        if patience >= 5: print("early stopping"); break
print(f"best val acc {best_acc:.4f} at epoch {best_epoch}")

## 5. Training curves

In [ ]:
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [h["train_loss"] for h in history], label="train"); ax[0].plot(ep, [h["val_loss"] for h in history], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, [h["train_acc"] for h in history], label="train"); ax[1].plot(ep, [h["val_acc"] for h in history], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Final evaluation on the held-out test set (scored once)

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "best_model.pth", map_location=device))
_, test_acc, y_true, y_pred = evaluate(test_dl)
print(f"TEST ACCURACY: {test_acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=classes))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title(f"Test confusion matrix (accuracy {test_acc:.3f})"); plt.tight_layout(); plt.show()

In [ ]:
rep = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
x = np.arange(len(classes)); w = 0.38
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, [rep[c]["precision"] for c in classes], w, label="precision", color="#2f7d46")
plt.bar(x + w/2, [rep[c]["recall"] for c in classes], w, label="recall", color="#9bd0a8")
plt.xticks(x, classes, rotation=30); plt.ylim(0, 1.05); plt.title("Per-class precision and recall (test set)"); plt.legend()
plt.tight_layout(); plt.show()

## 7. Sample predictions (wrong ones in red)

In [ ]:
rng = random.Random(SEED); sample = rng.sample(range(len(test_i)), 15)
ds = ImageList(test_i, eval_tf)
model.eval()
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, k in zip(axes.ravel(), sample):
    x, y = ds[k]
    with torch.no_grad(): p = torch.softmax(model(x.unsqueeze(0).to(device)), 1)[0]
    pred = p.argmax().item()
    ax.imshow(Image.open(ds.items[k][0]).convert("RGB")); ax.axis("off")
    ax.set_title(f"actual: {classes[y]}\npred: {classes[pred]} ({p[pred]:.2f})", color="green" if pred == y else "red", fontsize=9)
plt.tight_layout(); plt.show()

## 8. Explainability: Grad-CAM (where the model looks)

In [ ]:
acts, grads = {}, {}
layer = model.layer4[-1]
layer.register_forward_hook(lambda m, i, o: acts.update(a=o))
layer.register_full_backward_hook(lambda m, gi, go: grads.update(g=go[0]))

def gradcam(x, cls=None):
    model.zero_grad(); out = model(x.unsqueeze(0).to(device))
    cls = out.argmax(1).item() if cls is None else cls
    out[0, cls].backward()
    w = grads["g"].mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * acts["a"]).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode="bilinear")[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), cls

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for j, k in enumerate(sample[:6]):
    x, y = ds[k]; cam, pred = gradcam(x)
    img = Image.open(ds.items[k][0]).convert("RGB").resize((224, 224))
    axes[0, j].imshow(img); axes[0, j].axis("off"); axes[0, j].set_title(classes[y], fontsize=9)
    axes[1, j].imshow(img); axes[1, j].imshow(cam, cmap="jet", alpha=0.45); axes[1, j].axis("off"); axes[1, j].set_title(f"pred: {classes[pred]}", fontsize=9)
plt.tight_layout(); plt.show()

## 9. Save weights

In [ ]:
(OUT_DIR / "classes.json").write_text(json.dumps(classes))
(OUT_DIR / "metrics.json").write_text(json.dumps(dict(test_acc=test_acc, best_val_acc=best_acc, best_epoch=best_epoch, history=history), indent=2))
print("saved to", OUT_DIR, os.listdir(OUT_DIR))
try:
    from google.colab import files
    shutil.make_archive("/content/weights", "zip", OUT_DIR); files.download("/content/weights.zip")
except ImportError:
    pass